In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run /Workspace/Users/chrknov6@hotmail.com/formula1/Incremental/00.Configurations

In [0]:
%run "/Workspace/Users/chrknov6@hotmail.com/formula1/Incremental/003.gold helper functions"

In [0]:
from pyspark.sql.functions import col,lit

In [0]:
silver_table_circuits = f"{catalog}.{silver_schema}.circuits"

In [0]:
silver_table_races = f"{catalog}.{silver_schema}.races"

In [0]:
gold_table = f"{catalog}.{gold_schema}.dim_races"

In [0]:
circuits_df = spark.table(silver_table_circuits).filter(col("batch_id") == lit(v_batch_id))

In [0]:
races_df = spark.table(silver_table_races).filter(col("batch_id") == lit(v_batch_id))

In [0]:
final_df =(
           races_df.alias("r").join(circuits_df.alias("c"),"circuit_id","left")
                   .select(col("r.season"),col("r.round"),col("r.race_date"),col("r.race_name"),col("c.circuit_name"),col("c.locality"),col("c.country"))
          )


In [0]:
write_to_gold(
    input_df= final_df,
    table_name = gold_table,
    merge_condition="t.season = s.season AND t.round = s.round",
    columns_to_update=["season","round","race_date","race_name","circuit_name","locality","country"]
)